# 06 — Publish Outputs to ArcGIS Online

This notebook packages and publishes:

- The complete revised inspection-priority layer
- The simulated Year 1 inspection program.

The output descriptions explicitly disclose:
- DBH imputation
- DBH-derived maturity classes
- Simulated condition and inspection history
- The arterial-street exposure criterion
- The fact that the model does not predict tree failure

In [ ]:
from pathlib import Path
import os
from getpass import getpass
import shutil
import arcpy
from arcgis.gis import GIS

arcpy.env.overwriteOutput = True

PROJECT_ROOT = Path(os.environ.get("SEATTLE_TREE_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
GDB = PROJECT_ROOT / "data" / "Seattle_Tree_Assets.gdb"
PUBLISH_DIR = PROJECT_ROOT / "publish"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

TREES_PRIORITY = str(GDB / "trees_priority")
YEAR1_TREES = str(GDB / "inspection_program_year1")

for feature_class in [TREES_PRIORITY, YEAR1_TREES,]:
    if not arcpy.Exists(feature_class):
        raise FileNotFoundError(feature_class)

PORTAL_URL = "https://www.arcgis.com"
USERNAME = input("ArcGIS Online username: ").strip()
PASSWORD = getpass("ArcGIS Online password: ")

gis = GIS(PORTAL_URL, USERNAME, PASSWORD)
print(f"Signed in as: {gis.users.me.fullName}")

In [ ]:
PROJECT_TITLE = ("Seattle Urban Forestry Asset Management")
FOLDER_NAME = ("Seattle Tree Asset Management")
TAGS = ["Seattle", "urban forestry", "asset management", "inspection prioritization", "arterial streets", "municipal GIS", "portfolio project",]
DISCLAIMER = (
    "Portfolio demonstration using public City of Seattle data."
    "Missing DBH values were imputed using documented median-based methods."
    "Maturity classes were derived from DBH and do not represent verified age."
    "Condition and inspection-history fields are simulated."
    "This model prioritizes inspection workload and must not be interpreted as a professional tree-risk or failure assessment."
)

try:
    gis.content.folders.create(FOLDER_NAME)
except Exception:
    pass

In [ ]:
from pathlib import Path
import zipfile
import arcpy


def package_feature_class(feature_class, output_name,):
    """
    Copy one feature class into a clean file geodatabase and create
    a ZIP archive while excluding ArcGIS lock files.
    """

    package_gdb = PUBLISH_DIR / f"{output_name}.gdb"
    zip_path = PUBLISH_DIR / f"{output_name}.zip"

    # Remove previous outputs.
    if arcpy.Exists(str(package_gdb)):
        arcpy.management.Delete(str(package_gdb))

    if zip_path.exists():
        zip_path.unlink()

    # Create a clean publishing geodatabase.
    arcpy.management.CreateFileGDB(out_folder_path=str(PUBLISH_DIR), out_name=package_gdb.name,)

    output_fc = str(package_gdb / output_name)

    arcpy.management.CopyFeatures(in_features=feature_class, out_feature_class=output_fc,)

    copied_count = int(arcpy.management.GetCount(output_fc)[0])

    print(f"Copied {copied_count:,} records to "
        f"{output_fc}")

    # Release ArcGIS workspace references where possible.
    arcpy.management.ClearWorkspaceCache()

    # Create the ZIP manually and ignore transient lock files.
    with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED,) as archive:

        for file_path in package_gdb.rglob("*"):
            if not file_path.is_file():
                continue

            # Never publish ArcGIS lock files.
            if file_path.name.lower().endswith(".lock"):
                print( f"Skipping lock file: "
                    f"{file_path.name}")
                continue

            archive_name = file_path.relative_to(package_gdb.parent)

            archive.write(filename=file_path, arcname=archive_name,)

    if not zip_path.exists():
        raise RuntimeError(f"ZIP archive was not created: {zip_path}")

    print(f"Created publishing archive: {zip_path}")

    return zip_path

def find_existing_item(title, item_type,):
    results = gis.content.search(query=(f'title: "{title}" '  f'owner:{gis.users.me.username}'),item_type=item_type, max_items=20,)
    return next((item for item in results if item.title == title), None,)
    

In [ ]:
def publish_fgdb(zip_path, title, snippet,):
    existing_service = find_existing_item(title, "Feature Service",)

    if existing_service:
        print("Existing hosted layer found: \n", existing_service.title,)
        return existing_service
    
    item_properties = {
        "title": title,
        "type": "File Geodatabase",
        "tags": ",".join(TAGS),
        "snippet": snippet,
        "description":(
            f"{snippet}<br><br>"
            f"<strong>Methodology and limitations:</strong> "
            f"{DISCLAIMER}"),}
    
    folder = gis.content.folders.get(folder=FOLDER_NAME)
    uploaded = folder.add(
        item_properties=item_properties,
        file=str(zip_path),).result()
    published = uploaded.publish()
    print("Published:", published.title)
    return published

In [ ]:
all_zip = package_feature_class(TREES_PRIORITY, "SEATTLE_TREE_PRIORITY_v2",)
year1_zip =  package_feature_class(YEAR1_TREES, "Seattle_Tree_Inspection_Year1_v2",)

all_layer = publish_fgdb(
    all_zip, ("Seattle Tree Inspection Priority — Portfolio Simmulation v2"),
    ("Municipal tree assets scored using observed and imputed DBH, "
    "DBH-derived maturity, simulated condition and inspection history, "
    "and proximity to schools, sidewalks, trails, and arterial strrets."), )

year1_layer = publish_fgdb(year1_zip, 
                         ("Seattle Tree Inspection Program Year 1 "
                         "— Portfolio Simulation v2"),
                         ("The highest-priority five percent of municipal tree assets "
                         "selected for a simulated annual inspection program."), )


In [ ]:
SHARE_PUBLICLY = False

for item in [all_layer, year1_layer,]:
    item.update({
        "accessInformation" : ("City of Seattle Open Data; "
                              "portfolio analysis and simulation fields by project author."),
        "licenseInfo" : ("Sourse datasets retain their original terms. "
                        "Analysis ouputs are for demonstration only."),
    })

    if SHARE_PUBLICLY:
        item.sharing.sharing_level = ("EVERYONE")

    print(item.title)
    print(item.homepage)

## Manual ArcGIS Online finishing checklist

1. Symbolize `PRIOR_CLASS`.
2. Configure popups showing:
   - Asset ID
   - Park
   - Common and scientific name
   - `DBH_ORIG`
   - `DBH_CLEAN`
   - `DBH_IMPUTED`
   - `DBH_METHOD`
   - `MATURITY`
   - Simulated condition
   - Priority score
   - Year 1 selection
   - Model version
3. Include a visible note that maturity is DBH-derived, not verified age.
4. Include a visible note that condition and inspection history are simulated.
5. Create a Web Map using the full and Year 1 layers.
6. Build Dashboard indicators for total assets, Year 1 assets, estimated hours, high-priority assets, and imputed DBH count.
7. Add charts for priority class, maturity class, and park workload.
8. Use “arterial street proximity", in labels and narrative.
9. Keep the complete disclaimer in the item details, Dashboard, Experience Builder app, and StoryMap.